# Squirrel multiclass detection — YOLO26 Medium (v2)

Complete workflow: audited dataset → immutable YOLO conversion → model/GPU verification → memory probe → training → validation/test reports → CPU/GPU benchmarks → model package.

**Task:** eleven-class bounding-box detection, with ten species plus `generic_squirrel`.
**Recipe:** pretrained medium model, 640 pixels, standard native YOLO sampling/augmentations, seed 42, AMP, **at most 200 epochs and early-stopping patience 30**.

Run `uv sync --locked --all-extras` before selecting the project `.venv` kernel. Run cells in order. The training cell starts the full experiment; preparation and probe cells do not. Generated files stay under `output_yolo26/runs/<run-id>/`.

Recover an interrupted experiment by setting `RESUME_RUN` below and keeping its configuration unchanged. Complete checkpoints remain frozen for validation/test. Source COCO images and splits are never modified.

YOLO26 reports both its accuracy-oriented and NMS-free heads from independent loads of the same checkpoint. Original YOLOv5 uses `yolov5m.pt`, not the anchor-free YOLOv5u variant.

In [1]:
# 1. Nonsecret configuration and independent run folder.
from pathlib import Path
import json
from IPython.display import display, HTML, IFrame
from csci_635_ml_project.baseline import (
    default_config, create_run, prepare_run, probe_batches, train_run,
    predict_run, benchmark_run, package_run,
)
from csci_635_ml_project.baseline_reports import dataset_report, evaluate_run

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DATA = ROOT / "data/squirrel-v2-audited-coco"
CFG = default_config("yolo26")  # epochs=200, patience=30, imgsz=640, seed=42
RESUME_RUN = None  # ROOT / "output_yolo26/runs/<run-id>"
RUN = create_run(ROOT, CFG, resume_run=RESUME_RUN, data=DATA)
print("Run:", RUN)
display(CFG)

Run: /home/ss3813/github/ML/CSCI-635-ML-Project/output_yolo26/runs/20260928T202126Z-e14d4f


{'model': 'yolo26',
 'epochs': 200,
 'patience': 30,
 'imgsz': 640,
 'seed': 42,
 'batch_candidates': [16, 8, 4],
 'workers': 4,
 'amp': True,
 'save_period': 10,
 'optimizer': 'auto',
 'prediction_floor': 0.0001,
 'max_detections': 100,
 'matching_iou': 0.5,
 'nms_iou': 0.7,
 'heads': ['accuracy', 'end2end'],
 'ultralytics_version': '8.4.164',
 'yolov5_commit': '402e17ddf820996f51a191cbb798376e1144f069',
 'benchmark_images': 200,
 'benchmark_warmup': 20,
 'cpu_threads': 1}

## 2. Dataset verification, EDA, and backend setup

Require the passing audited v3 manifest and matching annotation hashes. Export the exact splits and class mapping to a fingerprinted YOLO dataset; every subsequent use verifies its image and label hashes. Native caches use the converted copy. Backend setup pins the model/source version, records pretrained weights and environment hashes, and executes a CUDA kernel.

In [2]:
DATA_YAML = prepare_run(RUN)
class_counts = dataset_report(RUN, DATA)
display(class_counts)
print("YOLO dataset:", DATA_YAML)
display(IFrame(src=str((RUN / "reports/report.html").relative_to(ROOT)), width="100%", height=700))

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/ss3813/github/ML/CSCI-635-ML-Project/output_yolo26/runs/20260928T202126Z-e14d4f/.ultralytics/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


,split,class_name,images,annotations
0,train,Callosciurus erythraeus,501,533
1,train,Sciurus aureogaster,534,570
2,train,Sciurus carolinensis,666,692
3,train,Sciurus granatensis,547,566
4,train,Sciurus griseus,581,655
5,train,Sciurus lis,409,449
6,train,Sciurus niger,551,606
7,train,Sciurus vulgaris,529,564
8,train,Tamiasciurus douglasii,574,591
9,train,Tamiasciurus hudsonicus,566,589


YOLO dataset: /home/ss3813/github/ML/CSCI-635-ML-Project/data/squirrel-v2-audited-yolo/96af70a0474fdebe/data.yaml


## 3. GPU memory probe

Try batches 16, 8, then 4 on a small seeded train/validation sample. Only CUDA OOM triggers a smaller-batch retry. Native nominal batch size is 64; resolved settings are saved. Probe weights are discarded for full training. Probe folders include batch-loss traces and a short Torch profiler trace where supported.

In [3]:
selected = probe_batches(RUN)
display(selected)

USDT:2026-09-28 16:22:18 2220647:2220647 SyncActivityProfilerHandler.cpp:39] profiler_start
New https://pypi.org/project/ultralytics/8.4.165 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.164 🚀 Python-3.13.15 torch-2.14.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 15938MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ss3813/github/ML/CSCI-635-ML-Project/output_yolo26/runs/20260928T202126Z-e14d4f/probe_data/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freez

{'batch_size': 16,
 'nominal_batch': 64,
 'attempts': [{'batch': 16, 'status': 'passed'}],
 'peak_vram_gib': 8.311372756958008}

## 4. Full training — this cell starts the experiment

Use the native optimizer/augmentation recipe with up to 200 epochs and 30-epoch early-stopping patience. YOLO26 selects its checkpoint by validation mAP50–95; original YOLOv5 uses 0.1 × AP50 + 0.9 × mAP50–95. Keep best/latest, periodic, and unstripped recovery checkpoints. Training logs, batch losses, epoch metrics, timing, VRAM, and loss curves stay in this run.

In [4]:
if (RUN / "training_complete.json").exists():
    print("Training already completed; using the frozen best checkpoint.")
else:
    history = train_run(RUN)
    display(history.tail())

New https://pypi.org/project/ultralytics/8.4.165 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.164 🚀 Python-3.13.15 torch-2.14.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 15938MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ss3813/github/ML/CSCI-635-ML-Project/data/squirrel-v2-audited-yolo/96af70a0474fdebe/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.0

,epoch,time,train/box_loss,train/cls_loss,train/l1_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,...,lr/pg0,lr/pg1,lr/pg2,lr/pg3,lr/pg4,lr/pg5,lr/pg6,lr/pg7,seconds,peak_vram_gib
135,136,12626.6,0.62739,0.45424,0.00694,0.76361,0.68495,0.72526,0.53427,0.95978,...,0.009952,0.003318,0.009952,0.003318,0.009952,0.003318,0.009952,0.003318,92.029656,8.415524
136,137,12718.2,0.62313,0.44111,0.00685,0.77313,0.68264,0.72843,0.53479,0.96182,...,0.009804,0.003268,0.009804,0.003268,0.009804,0.003268,0.009804,0.003268,91.608435,8.417042
137,138,12810.6,0.61311,0.43786,0.00672,0.78012,0.67502,0.72694,0.53428,0.95916,...,0.009655,0.003219,0.009655,0.003219,0.009655,0.003219,0.009655,0.003219,92.436721,8.414891
138,139,12903.1,0.60941,0.43801,0.00671,0.75794,0.69310,0.72774,0.53530,0.95841,...,0.009507,0.003169,0.009507,0.003169,0.009507,0.003169,0.009507,0.003169,92.472766,8.418847
139,140,12995.8,0.61319,0.43642,0.00668,0.76822,0.68017,0.72628,0.53560,0.96203,...,0.009359,0.003119,0.009359,0.003119,0.009359,0.003119,0.009359,0.003119,92.739680,8.415326


## 5. Common validation and held-out test evaluation

Predict at confidence floor 0.0001 with maximum 100 detections. Select each head's threshold using validation micro F1 at matching IoU 0.50; ties choose the highest threshold. Freeze its checkpoint/hash and threshold before test scoring. Final AP/AR uses the same pycocotools evaluator as RF-DETR, rather than comparing framework-native AP implementations.

Reports include per-class AP/precision/recall/F1, aggregate metrics, PR/confidence curves, raw and normalized confusion matrices including background, and annotated successes/errors. Wrong species contributes one false positive and one false negative. Loss scales are model-specific.

In [5]:
predict_run(RUN)
metrics = evaluate_run(RUN, DATA, heads=CFG["heads"])
for head, splits in metrics.items():
    print(head, {split: result["coco"]["mAP_50_95"] for split, result in splits.items()})
print("Reports:", RUN / "reports")

accuracy/valid: 200/1282 images
accuracy/valid: 400/1282 images
accuracy/valid: 600/1282 images
accuracy/valid: 800/1282 images
accuracy/valid: 1000/1282 images
accuracy/valid: 1200/1282 images
accuracy/test: 200/1284 images
accuracy/test: 400/1284 images
accuracy/test: 600/1284 images
accuracy/test: 800/1284 images
accuracy/test: 1000/1284 images
accuracy/test: 1200/1284 images
end2end/valid: 200/1282 images
end2end/valid: 400/1282 images
end2end/valid: 600/1282 images
end2end/valid: 800/1282 images
end2end/valid: 1000/1282 images
end2end/valid: 1200/1282 images
end2end/test: 200/1284 images
end2end/test: 400/1284 images
end2end/test: 600/1284 images
end2end/test: 800/1284 images
end2end/test: 1000/1284 images
end2end/test: 1200/1284 images
accuracy {'valid': 0.528389424456416, 'test': 0.5312803908581714}
end2end {'valid': 0.5468580509703936, 'test': 0.5483547564197625}
Reports: /home/ss3813/github/ML/CSCI-635-ML-Project/output_yolo26/runs/20260928T202126Z-e14d4f/reports


## 6. Local GPU and CPU latency

Benchmark identical seeded validation images, batch one, FP32, 20 warm-up calls, and one CPU thread. Include preprocessing/inference/postprocessing; exclude decoding. GPU timing synchronizes CUDA. No compilation or TF32. Save every image timing, mean/median/p95, FPS, hardware, checkpoint, threshold, and timing protocol. These measure local PyTorch performance; deployment-device timing requires a separate experiment.

In [6]:
benchmarks = benchmark_run(RUN)
import pandas as pd
display(pd.DataFrame([{k: row[k] for k in ("head", "device", "mean_ms", "median_ms", "p95_ms", "throughput_fps")} for row in benchmarks]))

accuracy/cuda: mean 23.91 ms, p95 24.61 ms
accuracy/cpu: mean 116.48 ms, p95 127.25 ms
end2end/cuda: mean 23.56 ms, p95 24.30 ms
end2end/cpu: mean 127.92 ms, p95 141.53 ms


,head,device,mean_ms,median_ms,p95_ms,throughput_fps
0,accuracy,cuda,23.911235,23.974351,24.613658,41.821345
1,accuracy,cpu,116.484016,116.053127,127.245818,8.584869
2,end2end,cuda,23.564081,23.558683,24.295340,42.437470
3,end2end,cpu,127.923669,126.111658,141.533559,7.817162


## 7. Package and optional explicit tracking

Export verified native best weights, class order, frozen thresholds, checkpoint/dataset hashes, model card, and report archive. Automatic framework tracking is disabled. If ClearML credentials exist privately in `.env`, this cell uploads explicit reports and metrics. Failures preserve local reports and a retry queue; no credentials are written into notebook outputs.

In [7]:
best_weights = package_run(RUN)
print("Verified best weights:", best_weights)
print("Recovery checkpoint:", RUN / "native/weights/last_resumable.pt")
print("Report archive:", RUN / "reports.zip")

ClearML Task: created new task id=2a551c380cc54b9888115b0ef64d4dc7
ClearML results page: https://app.clear.ml/projects/11ca3d88c4fe47ff86ffb8ecccdc8142/tasks/2a551c380cc54b9888115b0ef64d4dc7/output/log


███████████████████████████████ 100% | 19.37/19.37 MB [00:00<00:00, 55.24MB/s]: 


Verified best weights: /home/ss3813/github/ML/CSCI-635-ML-Project/output_yolo26/runs/20260928T202126Z-e14d4f/weights/best.pt
Recovery checkpoint: /home/ss3813/github/ML/CSCI-635-ML-Project/output_yolo26/runs/20260928T202126Z-e14d4f/native/weights/last_resumable.pt
Report archive: /home/ss3813/github/ML/CSCI-635-ML-Project/output_yolo26/runs/20260928T202126Z-e14d4f/reports.zip


## References

- [YOLO26 models and inference heads](https://docs.ultralytics.com/models/yolo26/)
- [Original YOLOv5 source](https://github.com/ultralytics/yolov5)
- [YOLOv5 versus YOLOv5u compatibility](https://docs.ultralytics.com/models/yolov5/)
- [Ultralytics training settings](https://docs.ultralytics.com/modes/train/)